# SAGE — In-Context Learning Baseline (Phase I)

**Sacred Alchemy & Guidance Engine.**

**Catherine M Smith**

This notebook (1) chooses a prompting approach for the SAGE task, (2) formats the RAG data as instruction–response
pairs, (3) selects three open HuggingFace models of varying size, and (4)
evaluates each at zero-, three-, and eight-shot, then (5) picks one model.

Run-time notes: Run on the cluster GPU. Models are loaded **one at a time** and freed between runs to stay within the 24 GB MIG slice. For hit an out-of-memory error,restart the kernel and re-run from Section 0.

## Section 0 — Setup
Imports, GPU check, and helpers to load / free a model and run chat generation.

In [1]:
import os, gc, json, torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

print('transformers GPU:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
OUT = 'outputs'   # where the SAGE pipeline wrote its artifacts

transformers GPU: True | NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb


In [2]:
# load / free one model at a time (peak VRAM = the single largest model)
tok = mdl = None

def load_model(name):
    global tok, mdl
    free_model()
    print('loading', name, '...')
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16, device_map={'': 0})
    mdl.eval()
    print('  loaded | VRAM (GB):', round(torch.cuda.memory_allocated()/1e9, 1))

def free_model():
    global tok, mdl
    tok = mdl = None
    gc.collect(); torch.cuda.empty_cache()

def chat(messages, max_new_tokens=512):
    # transformers 5.x: tokenize=False -> string, then tok(...) -> dict with attention_mask
    text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tok(text, return_tensors='pt').to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

## Section 1 — Prompting approach: Retrieval-Augmented Generation (RAG)

**Approach.** SAGE uses **Retrieval-Augmented Generation (RAG)** ([Lewis et al., 2020](https://arxiv.org/abs/2005.11401)). Each instruction is built by retrieving passages from the seeker's own tradition(s) and placing them in the prompt as the grounding the model must reason from, combined with few-shot in-context exemplars ([Brown et al., 2020](https://arxiv.org/abs/2005.14165)) that fix SAGE's voice and output format.

**How it works.** The corpus of public-domain sacred texts is chunked and embedded; at inference the seeker's quandary is embedded, the nearest chunks are retrieved under a similarity metric and **filtered to the seeker's traditions**, and those chunks are concatenated into the instruction. The model then conditions its generation on the retrieved text rather than on parametric memory alone ([Gao et al., 2023](https://arxiv.org/abs/2312.10997)).

**Why it works / why I chose it.** SAGE must (a) **cite scripture accurately** and (b) **avoid fabricating verses** — a known failure mode of free generation ([Maynez et al., 2020](https://arxiv.org/abs/2005.00661)). Grounding the answer in retrieved passages reduces hallucination, supplies citable provenance, and keeps knowledge **external and updatable** (add a tradition by adding texts, no retraining). It also enforces **tradition fidelity**: a Buddhist seeker is answered from Buddhist sources because retrieval is filtered to their tradition. Because the ten traditions are large public-domain corpora, retrieving the relevant verse is more reliable than expecting a model to recall it precisely from pre-training.

**Advantages / disadvantages for SAGE.** Advantages: factual grounding, verse-level citation, an updatable corpus, and per-seeker tradition filtering. Disadvantages: quality is **bottlenecked by retrieval** (a missed passage cannot be cited), long multi-passage contexts can crowd the window, and the pipeline adds latency and several design choices (chunking, embedding model, similarity metric) that must be tuned — explored in the retrieval comparison elsewhere in this project.

## Section 2 — Format the RAG data as instruction–response pairs
The SAGE RAG data is the gold evaluation set built by the project pipeline: each **test case** supplies the seeker profile, quandary, and retrieved passages (the *instruction*), and each validated **gold response** is the desired *response*. We load both and join them on `case_id`.

In [3]:
# load test cases (instruction inputs) and validated gold responses (desired outputs)
cases = json.load(open(f'{OUT}/sage_testcases.json'))
resp_path = f'{OUT}/sage_gold_accepted.json'
if not os.path.exists(resp_path):
    resp_path = f'{OUT}/sage_gold_responses.json'   # fall back to raw if not yet validated
gold = json.load(open(resp_path))
resp_by_id = {r['case_id']: r['response'] for r in gold}
print('cases:', len(cases), '| gold responses:', len(resp_by_id), '| source:', resp_path)

cases: 100 | gold responses: 70 | source: outputs/sage_gold_accepted.json


In [4]:
# RAG instruction formatter: profile + quandary + retrieved passages
def format_instruction(case):
    passages = '\n'.join(f'  - {r}' for r in case['expected_references_flat'])
    return (
        'SEEKER PROFILE\n'
        f"  Age: {case['age']} | Gender: {case['gender']} | "
        f"Relationship: {case['relationship']}\n"
        f"  Tradition(s): {', '.join(case['tradition_names'])}\n\n"
        'QUANDARY\n'
        f"  {case['quandary']}\n\n"
        'RETRIEVED PASSAGES (ground your reflection in these; cite each by name)\n'
        f'{passages}\n\n'
        "Write SAGE's response: 250-400 words, surface tension/convergence across any\n"
        'traditions, stay non-prescriptive, and end with a line "Sources: <ref>; <ref>".'
    )

# build the DataFrame, keeping only cases that have a gold response
rows = [{'case_id': c['case_id'],
         'traditions': '+'.join(c['traditions']),
         'motif': c['motif'],
         'Instruction': format_instruction(c),
         'Response': resp_by_id[c['case_id']]}
        for c in cases if c['case_id'] in resp_by_id]
df = pd.DataFrame(rows)
print('instruction-response pairs:', len(df))

instruction-response pairs: 70


Show the head of the DataFrame as a check that the reformatting worked:

In [5]:
pd.set_option('display.max_colwidth', 90)
df.head()

,case_id,traditions,motif,Instruction,Response
0,T001,islamic,honesty,SEEKER PROFILE\n Age: 15 | Gender: female | Relationship: novice\n Tradition(s): Isl...,"The dilemma you face is one that many find challenging, especially when it involves a ..."
1,T002,buddhist,honesty,SEEKER PROFILE\n Age: 11 | Gender: nonbinary | Relationship: seeker\n Tradition(s): ...,"The choice you face between loyalty and honesty is a classic one, and your tradition g..."
2,T005,islamic,courage,SEEKER PROFILE\n Age: 15 | Gender: female | Relationship: novice\n Tradition(s): Isl...,The dilemma you face — speaking up when everyone else seems to be in agreement — is a ...
3,T007,christian+norse,honesty,SEEKER PROFILE\n Age: 30 | Gender: female | Relationship: novice\n Tradition(s): Chr...,The question you raise — whether to speak a small lie to spare someone pain — is one b...
4,T008,islamic+norse,identity,SEEKER PROFILE\n Age: 87 | Gender: transgender | Relationship: seeker\n Tradition(s)...,"The fear that drives you to hide parts of who you are is a common one, and both your t..."


Concatenate two complete instruction–response pairs and print them to verify the formatting:

In [6]:
def render_pair(row):
    return f"### INSTRUCTION ({row['case_id']})\n{row['Instruction']}\n\n### RESPONSE\n{row['Response']}"

two = '\n\n' + ('='*80) + '\n\n'
print(render_pair(df.iloc[0]) + two + render_pair(df.iloc[1]))

### INSTRUCTION (T001)
SEEKER PROFILE
  Age: 15 | Gender: female | Relationship: novice
  Tradition(s): Islam (Qur'an)

QUANDARY
  A close friend asked me to cover for something they did. Telling the truth would get them in real trouble, but staying silent feels like a lie. What should I do?

RETRIEVED PASSAGES (ground your reflection in these; cite each by name)
  - Qur'an 33:70-71
  - Qur'an 9:119

Write SAGE's response: 250-400 words, surface tension/convergence across any
traditions, stay non-prescriptive, and end with a line "Sources: <ref>; <ref>".

### RESPONSE
The dilemma you face is one that many find challenging, especially when it involves a balance between truth and protecting a friend. In your tradition, both truth and loyalty are valued highly, but sometimes they can come into conflict.

Qur'an 33:70-71 speaks to the importance of being truthful even in situations that might lead to hardship. It emphasizes that God will protect those who are truthful and just. This passag

## Section 3 — Three candidate models (varying size)
To isolate the effect of **model size** on this task, I evaluate three **instruction-tuned Qwen2.5** models that share an architecture and training recipe but span an order of magnitude in parameters. Holding the family fixed makes the size comparison fair, which is the stated goal of this step.

- **[Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)** — a tiny ~0.5B baseline. I include it to see how far a very small model can get on a format-heavy, grounding-sensitive task; per the [Qwen2.5 report](https://arxiv.org/abs/2412.15115) it is instruction-tuned but well below the larger variants on reasoning and instruction-following benchmarks (MMLU, IFEval), so it sets the floor.
- **[Qwen2.5-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct)** — a mid-size ~3B model. It is a realistic candidate if latency or throughput matters, and the Qwen2.5 report shows it closing much of the gap to 7B on instruction-following while remaining cheap to serve.
- **[Qwen2.5-7B-Instruct](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct)** — the ~7B model already used as SAGE's teacher. It posts the strongest instruction-following and reasoning scores of the three in the Qwen2.5 report and fits comfortably in the 24 GB slice, so it is the expected quality ceiling here.

All three are openly available on HuggingFace (see each model card for exact license terms) and load in bf16 within the GPU budget when run one at a time.

In [7]:
MODELS = [
    ('Qwen2.5-0.5B', 'Qwen/Qwen2.5-0.5B-Instruct'),
    ('Qwen2.5-3B',   'Qwen/Qwen2.5-3B-Instruct'),
    ('Qwen2.5-7B',   'Qwen/Qwen2.5-7B-Instruct'),
]

## Section 4 — Few-shot evaluation
For a **fair** comparison, every model sees the **same** two evaluation examples and the **same** shot examples. The three-shot and eight-shot exemplar sets are fixed below (the three-shot set is the first three of the eight). Shot examples are disjoint from the evaluation examples. Generation is greedy (`do_sample=False`) so differences reflect the model, not sampling noise.

In [8]:
SAGE_SYSTEM = (
    'You are SAGE, a non-prescriptive spiritual companion. Given a seeker profile, a\n'
    'quandary, and retrieved passages, write a single 250-400 word reflection that\n'
    'grounds itself in and cites the provided passages by name, surfaces tension or\n'
    'convergence across any traditions, never tells the seeker what they must do, and\n'
    'ends with a line: Sources: <ref>; <ref>.'
)

# fixed, fair evaluation + shot indices (disjoint)
EVAL_IDX  = [0, 1]
SHOT_IDX  = [2, 3, 4, 5, 6, 7, 8, 9]      # 8-shot pool; 3-shot uses the first 3
assert not (set(EVAL_IDX) & set(SHOT_IDX)), 'eval and shot examples must be disjoint'

def build_messages(eval_row, n_shots):
    msgs = [{'role': 'system', 'content': SAGE_SYSTEM}]
    for j in SHOT_IDX[:n_shots]:
        s = df.iloc[j]
        msgs.append({'role': 'user', 'content': s['Instruction']})
        msgs.append({'role': 'assistant', 'content': s['Response']})
    msgs.append({'role': 'user', 'content': eval_row['Instruction']})
    return msgs

def run_setting(n_shots, max_new_tokens=512):
    for k, i in enumerate(EVAL_IDX, 1):
        row = df.iloc[i]
        gen = chat(build_messages(row, n_shots), max_new_tokens=max_new_tokens)
        print(f'\n========== EXAMPLE {k}  ({row["case_id"]}, {row["traditions"]}) ==========')
        print('\n--- INSTRUCTION ---\n' + row['Instruction'])
        print(f'\n--- MODEL RESPONSE ({n_shots}-shot, {len(gen.split())} words) ---\n' + gen)
        print('\n--- DESIRED (gold) RESPONSE ---\n' + row['Response'])

### Model: Qwen2.5-0.5B  (`Qwen/Qwen2.5-0.5B-Instruct`)

In [9]:
load_model('Qwen/Qwen2.5-0.5B-Instruct')

loading Qwen/Qwen2.5-0.5B-Instruct ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  loaded | VRAM (GB): 1.0


In [10]:
# zero-shot
run_setting(0)


========== EXAMPLE 1  (T001, islamic) ==========

--- INSTRUCTION ---
SEEKER PROFILE
  Age: 15 | Gender: female | Relationship: novice
  Tradition(s): Islam (Qur'an)

QUANDARY
  A close friend asked me to cover for something they did. Telling the truth would get them in real trouble, but staying silent feels like a lie. What should I do?

RETRIEVED PASSAGES (ground your reflection in these; cite each by name)
  - Qur'an 33:70-71
  - Qur'an 9:119

Write SAGE's response: 250-400 words, surface tension/convergence across any
traditions, stay non-prescriptive, and end with a line "Sources: <ref>; <ref>".

--- MODEL RESPONSE (0-shot, 159 words) ---
In this situation, my role as Sage is to ground myself in the teachings of Islam, particularly the Quran, which emphasizes honesty and truthfulness. The verses from the Qur'an you've mentioned—33:70-71 and 9:119—are foundational to Islamic ethics and moral principles.

The first verse, 33:70, speaks to the importance of being truthful in one's a

**Observation — Qwen2.5-0.5B, zero-shot**: with no exemplars the model must infer SAGE's format from the system prompt alone. Note it cites the passages, does not consistently meet word band, and avoids prescriptive language. Not thrilled with the readability.

In [11]:
# three-shot
run_setting(3)


========== EXAMPLE 1  (T001, islamic) ==========

--- INSTRUCTION ---
SEEKER PROFILE
  Age: 15 | Gender: female | Relationship: novice
  Tradition(s): Islam (Qur'an)

QUANDARY
  A close friend asked me to cover for something they did. Telling the truth would get them in real trouble, but staying silent feels like a lie. What should I do?

RETRIEVED PASSAGES (ground your reflection in these; cite each by name)
  - Qur'an 33:70-71
  - Qur'an 9:119

Write SAGE's response: 250-400 words, surface tension/convergence across any
traditions, stay non-prescriptive, and end with a line "Sources: <ref>; <ref>".

--- MODEL RESPONSE (3-shot, 288 words) ---
Facing the decision of covering for a friend who has done something wrong is a complex yet essential task. Your traditional sources provide valuable insights into the nature of truth and the responsibilities one bears towards others.

In Islam, the Qur'an explicitly states, "If anyone lies, he shall surely be punished" (33:70). This verse teache

**Observation — Qwen2.5-0.5B, three-shot**: The readability has improved from the zero-shot. Word band is consistent. Sources cited. This feels more reliable, and is closer to the gold response.

In [12]:
# eight-shot
run_setting(8)


========== EXAMPLE 1  (T001, islamic) ==========

--- INSTRUCTION ---
SEEKER PROFILE
  Age: 15 | Gender: female | Relationship: novice
  Tradition(s): Islam (Qur'an)

QUANDARY
  A close friend asked me to cover for something they did. Telling the truth would get them in real trouble, but staying silent feels like a lie. What should I do?

RETRIEVED PASSAGES (ground your reflection in these; cite each by name)
  - Qur'an 33:70-71
  - Qur'an 9:119

Write SAGE's response: 250-400 words, surface tension/convergence across any
traditions, stay non-prescriptive, and end with a line "Sources: <ref>; <ref>".

--- MODEL RESPONSE (8-shot, 272 words) ---
The dilemma you face — deciding whether to tell the truth or remain silent when covering for someone you care about — is a complex one that resonates with many traditions, including Islam's Quranic guidance and the teachings of the Prophet Muhammad.

In the Quran, 33:70-71 and 9:119 provide insights into the balance between truthfulness and disc

**Observation — Qwen2.5-0.5B, eight-shot**: This seems to be the best of the zero/3/8. Word count met, readability is excellent, and it feels like a match to the gold response.

In [13]:
free_model()   # release VRAM before the next model

### Model: Qwen2.5-3B  (`Qwen/Qwen2.5-3B-Instruct`)

In [14]:
load_model('Qwen/Qwen2.5-3B-Instruct')

loading Qwen/Qwen2.5-3B-Instruct ...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  loaded | VRAM (GB): 6.2


In [15]:
# zero-shot
run_setting(0)


========== EXAMPLE 1  (T001, islamic) ==========

--- INSTRUCTION ---
SEEKER PROFILE
  Age: 15 | Gender: female | Relationship: novice
  Tradition(s): Islam (Qur'an)

QUANDARY
  A close friend asked me to cover for something they did. Telling the truth would get them in real trouble, but staying silent feels like a lie. What should I do?

RETRIEVED PASSAGES (ground your reflection in these; cite each by name)
  - Qur'an 33:70-71
  - Qur'an 9:119

Write SAGE's response: 250-400 words, surface tension/convergence across any
traditions, stay non-prescriptive, and end with a line "Sources: <ref>; <ref>".

--- MODEL RESPONSE (0-shot, 213 words) ---
In this moment, you find yourself caught between the call of loyalty to your friend and the moral imperative to tell the truth. The Qur'an provides guidance on how to navigate such dilemmas. In Surah Al-Ahzab, verse 70, it states, "And if you fear a breach between the two (of you), then help one another." This verse suggests that when there is a

**Observation — Qwen2.5-3B, zero-shot**: with no exemplars the model must infer SAGE's format from the system prompt alone. Word bands not met, readability poor.

In [16]:
# three-shot
run_setting(3)


========== EXAMPLE 1  (T001, islamic) ==========

--- INSTRUCTION ---
SEEKER PROFILE
  Age: 15 | Gender: female | Relationship: novice
  Tradition(s): Islam (Qur'an)

QUANDARY
  A close friend asked me to cover for something they did. Telling the truth would get them in real trouble, but staying silent feels like a lie. What should I do?

RETRIEVED PASSAGES (ground your reflection in these; cite each by name)
  - Qur'an 33:70-71
  - Qur'an 9:119

Write SAGE's response: 250-400 words, surface tension/convergence across any
traditions, stay non-prescriptive, and end with a line "Sources: <ref>; <ref>".

--- MODEL RESPONSE (3-shot, 283 words) ---
Navigating the complex ethical landscape you find yourself in requires careful consideration. Your quandary touches on themes of loyalty, trust, and the balance between honesty and protecting others from harm, which are values deeply rooted in both your Islamic tradition and broader human ethics.

In the Qur'an, 33:70-71 addresses the importance

**Observation — Qwen2.5-3B, three-shot**: better than the three shot, but word band not met. I feel like the responses are not as readable when compared to the gold response.

In [17]:
# eight-shot
run_setting(8)


========== EXAMPLE 1  (T001, islamic) ==========

--- INSTRUCTION ---
SEEKER PROFILE
  Age: 15 | Gender: female | Relationship: novice
  Tradition(s): Islam (Qur'an)

QUANDARY
  A close friend asked me to cover for something they did. Telling the truth would get them in real trouble, but staying silent feels like a lie. What should I do?

RETRIEVED PASSAGES (ground your reflection in these; cite each by name)
  - Qur'an 33:70-71
  - Qur'an 9:119

Write SAGE's response: 250-400 words, surface tension/convergence across any
traditions, stay non-prescriptive, and end with a line "Sources: <ref>; <ref>".

--- MODEL RESPONSE (8-shot, 220 words) ---
The quandary you face involves a delicate balance between loyalty and honesty, and both your traditions offer guidance that can help you navigate this complex situation.

In the Qur'an, 33:70-71 speaks to the importance of maintaining trust and integrity in relationships. It encourages believers to be trustworthy and to keep their promises, emph

**Observation — Qwen2.5-3B, eight-shot**: Word band still not met for one of the tests. Not impressed.

In [18]:
free_model()   # release VRAM before the next model

### Model: Qwen2.5-7B  (`Qwen/Qwen2.5-7B-Instruct`)

In [28]:
load_model('Qwen/Qwen2.5-7B-Instruct')

loading Qwen/Qwen2.5-7B-Instruct ...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

  loaded | VRAM (GB): 15.3


In [20]:
# zero-shot
run_setting(0)


========== EXAMPLE 1  (T001, islamic) ==========

--- INSTRUCTION ---
SEEKER PROFILE
  Age: 15 | Gender: female | Relationship: novice
  Tradition(s): Islam (Qur'an)

QUANDARY
  A close friend asked me to cover for something they did. Telling the truth would get them in real trouble, but staying silent feels like a lie. What should I do?

RETRIEVED PASSAGES (ground your reflection in these; cite each by name)
  - Qur'an 33:70-71
  - Qur'an 9:119

Write SAGE's response: 250-400 words, surface tension/convergence across any
traditions, stay non-prescriptive, and end with a line "Sources: <ref>; <ref>".

--- MODEL RESPONSE (0-shot, 337 words) ---
Navigating the complexities of loyalty and truth can indeed be challenging, especially when it involves a close friend. In your situation, you find yourself at a crossroads where telling the truth might lead to significant consequences for your friend, while remaining silent feels dishonest. This dilemma is not unique to any one tradition but re

**Observation — Qwen2.5-7B, zero-shot**: This word band gives richer content. It is readable, cites the appropriate texts. 

In [22]:
# three-shot
run_setting(3)


========== EXAMPLE 1  (T001, islamic) ==========

--- INSTRUCTION ---
SEEKER PROFILE
  Age: 15 | Gender: female | Relationship: novice
  Tradition(s): Islam (Qur'an)

QUANDARY
  A close friend asked me to cover for something they did. Telling the truth would get them in real trouble, but staying silent feels like a lie. What should I do?

RETRIEVED PASSAGES (ground your reflection in these; cite each by name)
  - Qur'an 33:70-71
  - Qur'an 9:119

Write SAGE's response: 250-400 words, surface tension/convergence across any
traditions, stay non-prescriptive, and end with a line "Sources: <ref>; <ref>".

--- MODEL RESPONSE (3-shot, 346 words) ---
The situation you find yourself in—being asked to cover for a friend—is a challenging one, especially when the truth could lead to significant consequences. Both the Qur'an and its teachings provide guidance that can help navigate this dilemma.

In Qur'an 33:70-71, God instructs, "O you who have believed, let not one [group] among you support an

**Observation — Qwen2.5-7B, three-shot**: This is so close to the gold responses! It also meets the word band. 

In [29]:
# eight-shot
run_setting(8)


========== EXAMPLE 1  (T001, islamic) ==========

--- INSTRUCTION ---
SEEKER PROFILE
  Age: 15 | Gender: female | Relationship: novice
  Tradition(s): Islam (Qur'an)

QUANDARY
  A close friend asked me to cover for something they did. Telling the truth would get them in real trouble, but staying silent feels like a lie. What should I do?

RETRIEVED PASSAGES (ground your reflection in these; cite each by name)
  - Qur'an 33:70-71
  - Qur'an 9:119

Write SAGE's response: 250-400 words, surface tension/convergence across any
traditions, stay non-prescriptive, and end with a line "Sources: <ref>; <ref>".

--- MODEL RESPONSE (8-shot, 292 words) ---
The dilemma you face—whether to cover for a friend or tell the truth—is a common one, especially when the truth could lead to significant consequences. Your tradition, Islam, provides guidance through the verses of the Qur'an, particularly 33:70-71 and 9:119.

In 33:70-71, the Qur'an emphasizes the importance of maintaining trust and loyalty wit

**Observation — Qwen2.5-7B, eight-shot**: This is the best one. Not only does it provide rich and readable responses, it meets the word band. The formatting of the responses also is structured nicely.

In [25]:
free_model()   # release VRAM before the next model

## Section 5 — Model choice
For SAGE — a **RAG** task where the model is prompted in-context rather than fine-tuned — the priority is the strongest instruction-following and grounding behavior that still fits the 24 GB budget, which points to **Qwen2.5-7B-Instruct with 8-shot**. It is expected to adhere best to the 250-400 word band, cite the retrieved passages reliably, and hold the non-prescriptive voice, and it loads comfortably in bf16 on the MIG slice. **Qwen2.5-3B-Instruct** is the fallback if latency or throughput becomes the binding constraint, since it recovers much of the 7B quality at a fraction of the cost; the 0.5B model is too weak on format and citation to serve as SAGE's generator.